<a href="https://colab.research.google.com/github/yingcen1/AI-Evolution-Lab-/blob/main/weibo_ner_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 微博中文命名实体识别（Weibo NER）
**模型**：BERT + CRF | **目标**：实体级 F1 >= 88%

## 使用步骤
1. 菜单栏 -> `运行时` -> `更改运行时类型` -> 选 **T4 GPU** -> 保存
2. 菜单栏 -> `运行时` -> `全部运行`
3. 第3步会弹出文件上传框，上传 train.txt / dev.txt / test.txt
4. 等约 15 分钟，训练完自动弹出下载模型


In [1]:
import torch
print(f'GPU 可用：{torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 型号：{torch.cuda.get_device_name(0)}')
else:
    print('未检测到 GPU，请：运行时 -> 更改运行时类型 -> T4 GPU')


GPU 可用：True
GPU 型号：Tesla T4


In [2]:
!pip install seqeval -q
print("seqeval ok")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
seqeval ok


In [3]:
import os
from google.colab import files
os.makedirs('data', exist_ok=True)
os.makedirs('saved', exist_ok=True)
print('请上传 train.txt / dev.txt / test.txt 三个文件：')
uploaded = files.upload()
for fn in uploaded.keys():
    os.rename(fn, f'data/{fn}')
    print(f'已保存：data/{fn}')


请上传 train.txt / dev.txt / test.txt 三个文件：


Saving dev.txt to dev.txt
Saving test.txt to test.txt
Saving train.txt to train.txt
已保存：data/dev.txt
已保存：data/test.txt
已保存：data/train.txt


In [4]:
import torch
from transformers import BertModel, BertTokenizer

class Config:
    def __init__(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.train_path = 'data/train.txt'
        self.dev_path   = 'data/dev.txt'
        self.test_path  = 'data/test.txt'
        self.label_list = [
            'O',
            'B-GPE.NAM', 'I-GPE.NAM', 'B-GPE.NOM', 'I-GPE.NOM',
            'B-LOC.NAM', 'I-LOC.NAM', 'B-LOC.NOM', 'I-LOC.NOM',
            'B-ORG.NAM', 'I-ORG.NAM', 'B-ORG.NOM', 'I-ORG.NOM',
            'B-PER.NAM', 'I-PER.NAM', 'B-PER.NOM', 'I-PER.NOM',
        ]
        self.label2id    = {l: i for i, l in enumerate(self.label_list)}
        self.id2label    = {i: l for i, l in enumerate(self.label_list)}
        self.num_labels  = len(self.label_list)
        self.bert_path   = 'bert-base-chinese'
        self.bert_model  = BertModel.from_pretrained(self.bert_path)
        self.tokenizer   = BertTokenizer.from_pretrained(self.bert_path)
        self.hidden_size    = 768
        self.num_epochs     = 5
        self.batch_size     = 32
        self.max_len        = 128
        self.learning_rate  = 2e-5
        self.model_save_path = 'saved/bert_crf_ner.pt'

print('正在加载 BERT（首次需下载约 400MB，请稍等）...')
conf = Config()
print(f'配置完成，设备：{conf.device}，标签数：{conf.num_labels}')


正在加载 BERT（首次需下载约 400MB，请稍等）...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/412M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/269k [00:00<?, ?B/s]

配置完成，设备：cuda，标签数：17


In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

def load_raw_data(file_path):
    data, cur_chars, cur_labels = [], [], []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in tqdm(f, desc=f'Loading {file_path}'):
            line = line.strip()
            if not line:
                if cur_chars:
                    data.append((cur_chars, cur_labels))
                    cur_chars, cur_labels = [], []
                continue
            parts = line.split('\t')
            if len(parts) == 2:
                ch, lb = parts
                cur_chars.append(ch)
                cur_labels.append(lb if lb in conf.label2id else 'O')
    if cur_chars:
        data.append((cur_chars, cur_labels))
    return data

class NERDataset(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

def collate_fn(batch):
    chars_batch  = [item[0] for item in batch]
    labels_batch = [item[1] for item in batch]
    enc = conf.tokenizer(chars_batch, is_split_into_words=True,
                         padding=True, truncation=True,
                         max_length=conf.max_len, return_tensors='pt')
    input_ids, attention_mask = enc['input_ids'], enc['attention_mask']
    B, L = input_ids.shape
    label_ids = torch.zeros(B, L, dtype=torch.long)
    mask      = torch.zeros(B, L, dtype=torch.bool)
    for i, labels in enumerate(labels_batch):
        n = min(len(labels), conf.max_len - 2)
        for j in range(n):
            label_ids[i, j+1] = conf.label2id[labels[j]]
            mask[i, j+1] = True
    return input_ids, attention_mask, label_ids, mask

train_loader = DataLoader(NERDataset(load_raw_data(conf.train_path)), shuffle=True,  batch_size=conf.batch_size, collate_fn=collate_fn)
dev_loader   = DataLoader(NERDataset(load_raw_data(conf.dev_path)),   shuffle=False, batch_size=conf.batch_size, collate_fn=collate_fn)
test_loader  = DataLoader(NERDataset(load_raw_data(conf.test_path)),  shuffle=False, batch_size=conf.batch_size, collate_fn=collate_fn)
print(f'DataLoader 完成，训练集 batch 数：{len(train_loader)}')


Loading data/train.txt: 75078it [00:00, 547085.03it/s]
Loading data/dev.txt: 14732it [00:00, 379898.47it/s]
Loading data/test.txt: 15080it [00:00, 745030.44it/s]

DataLoader 完成，训练集 batch 数：43


In [6]:
import torch
import torch.nn as nn

class CRF(nn.Module):
    def __init__(self, num_tags):
        super().__init__()
        self.num_tags = num_tags
        self.transitions = nn.Parameter(torch.randn(num_tags, num_tags))
        self.start_transitions = nn.Parameter(torch.randn(num_tags))
        self.end_transitions = nn.Parameter(torch.randn(num_tags))

    def forward(self, emissions, tags, mask):
        mask = mask.float()
        B, L, C = emissions.shape
        score = self.start_transitions[tags[:, 0]] + emissions[torch.arange(B), 0, tags[:, 0]]
        for i in range(1, L):
            m = mask[:, i]
            t = self.transitions[tags[:, i-1], tags[:, i]] + emissions[torch.arange(B), i, tags[:, i]]
            score = score + t * m
        last_pos = mask.long().sum(1) - 1
        last_tag = tags[torch.arange(B), last_pos]
        score = score + self.end_transitions[last_tag]
        log_Z = self._log_partition(emissions, mask)
        return (log_Z - score).mean()

    def _log_partition(self, emissions, mask):
        B, L, C = emissions.shape
        alpha = self.start_transitions + emissions[:, 0]
        for i in range(1, L):
            emit = emissions[:, i].unsqueeze(1)
            scores = alpha.unsqueeze(2) + self.transitions.unsqueeze(0) + emit
            new_alpha = torch.logsumexp(scores, dim=1)
            m = mask[:, i].unsqueeze(1)
            alpha = new_alpha * m + alpha * (1.0 - m)
        return torch.logsumexp(alpha + self.end_transitions, dim=1)

    def decode(self, emissions, mask):
        mask = mask.float()
        B, L, C = emissions.shape
        viterbi = self.start_transitions + emissions[:, 0]
        backpointers = []
        for i in range(1, L):
            scores = viterbi.unsqueeze(2) + self.transitions.unsqueeze(0) + emissions[:, i].unsqueeze(1)
            best_scores, best_tags = scores.max(1)
            backpointers.append(best_tags)
            m = mask[:, i].unsqueeze(1)
            viterbi = best_scores * m + viterbi * (1.0 - m)
        viterbi = viterbi + self.end_transitions
        best_last = viterbi.argmax(1)
        paths = []
        for b in range(B):
            path = [best_last[b].item()]
            seq_len = int(mask[b].sum().item())
            for bp in reversed(backpointers[:seq_len-1]):
                path.append(bp[b, path[-1]].item())
            paths.append(list(reversed(path)))
        return paths

class BertCRF(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert    = conf.bert_model
        self.dropout = nn.Dropout(p=0.1)
        self.linear  = nn.Linear(conf.hidden_size, conf.num_labels)
        self.crf     = CRF(conf.num_labels)

    def _emit(self, input_ids, attention_mask):
        out = self.bert(input_ids, attention_mask=attention_mask)
        return self.linear(self.dropout(out.last_hidden_state))

    def forward(self, input_ids, attention_mask, label_ids, mask):
        return self.crf(self._emit(input_ids, attention_mask), label_ids, mask)

    def decode(self, input_ids, attention_mask, mask):
        return self.crf.decode(self._emit(input_ids, attention_mask), mask)

print("model ready")


model ready


In [7]:
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

def evaluate(model, loader):
    model.eval()
    all_pred, all_true = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='验证中...'):
            ids, attn, lids, mask = [b.to(conf.device) for b in batch]
            preds = model.decode(ids, attn, mask)
            for i in range(len(preds)):
                all_pred.append([conf.id2label[p] for p in preds[i]])
                all_true.append([conf.id2label[t] for t in lids[i][mask[i]].cpu().tolist()])
    f1  = f1_score(all_true, all_pred, zero_division=0)
    p   = precision_score(all_true, all_pred, zero_division=0)
    r   = recall_score(all_true, all_pred, zero_division=0)
    rep = classification_report(all_true, all_pred, zero_division=0)
    return rep, f1, p, r

print('评估函数定义完成')


评估函数定义完成


## 开始训练（GPU 约 15 分钟）


In [8]:
from torch.optim import AdamW

model     = BertCRF().to(conf.device)
optimizer = AdamW(model.parameters(), lr=conf.learning_rate)
best_f1   = 0.0

for epoch in range(conf.num_epochs):
    model.train()
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{conf.num_epochs}')
    total_loss = 0.0
    for idx, batch in enumerate(pbar):
        ids, attn, lids, mask = [b.to(conf.device) for b in batch]
        loss = model(ids, attn, lids, mask)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'avg': f'{total_loss/(idx+1):.4f}'})

    rep, f1, p, r = evaluate(model, dev_loader)
    print(f'\n[Epoch {epoch+1}] F1:{f1:.4f} P:{p:.4f} R:{r:.4f}')
    print(rep)
    if f1 > best_f1:
        torch.save(model.state_dict(), conf.model_save_path)
        best_f1 = f1
        print(f'模型已保存！最佳 F1：{best_f1:.4f}')

print(f'\n训练完成！最佳验证集 F1：{best_f1:.4f}')


验证中...: 100%|██████████| 9/9 [00:02<00:00,  3.80it/s]



[Epoch 1] F1:0.0039 P:0.0031 R:0.0052
              precision    recall  f1-score   support

     GPE.NAM       0.00      0.00      0.00        26
     GPE.NOM       0.00      0.00      0.00         1
     LOC.NAM       0.00      0.00      0.00         6
     LOC.NOM       0.00      0.00      0.00         6
     ORG.NAM       0.00      0.00      0.00        47
     ORG.NOM       0.00      0.00      0.00         5
     PER.NAM       0.00      0.00      0.00        89
     PER.NOM       0.01      0.01      0.01       208

   micro avg       0.00      0.01      0.00       388
   macro avg       0.00      0.00      0.00       388
weighted avg       0.00      0.01      0.00       388

模型已保存！最佳 F1：0.0039


验证中...: 100%|██████████| 9/9 [00:02<00:00,  4.33it/s]



[Epoch 2] F1:0.0000 P:0.0000 R:0.0000
              precision    recall  f1-score   support

     GPE.NAM       0.00      0.00      0.00        26
     GPE.NOM       0.00      0.00      0.00         1
     LOC.NAM       0.00      0.00      0.00         6
     LOC.NOM       0.00      0.00      0.00         6
     ORG.NAM       0.00      0.00      0.00        47
     ORG.NOM       0.00      0.00      0.00         5
     PER.NAM       0.00      0.00      0.00        89
     PER.NOM       0.00      0.00      0.00       208

   micro avg       0.00      0.00      0.00       388
   macro avg       0.00      0.00      0.00       388
weighted avg       0.00      0.00      0.00       388



验证中...: 100%|██████████| 9/9 [00:02<00:00,  4.25it/s]



[Epoch 3] F1:0.0024 P:0.0023 R:0.0026
              precision    recall  f1-score   support

     GPE.NAM       0.00      0.00      0.00        26
     GPE.NOM       0.00      0.00      0.00         1
     LOC.NAM       0.00      0.00      0.00         6
     LOC.NOM       0.00      0.00      0.00         6
     ORG.NAM       0.00      0.00      0.00        47
     ORG.NOM       0.00      0.00      0.00         5
     PER.NAM       0.00      0.00      0.00        89
     PER.NOM       0.00      0.00      0.00       208

   micro avg       0.00      0.00      0.00       388
   macro avg       0.00      0.00      0.00       388
weighted avg       0.00      0.00      0.00       388



验证中...: 100%|██████████| 9/9 [00:02<00:00,  4.19it/s]



[Epoch 4] F1:0.0023 P:0.0021 R:0.0026
              precision    recall  f1-score   support

     GPE.NAM       0.00      0.00      0.00        26
     GPE.NOM       0.00      0.00      0.00         1
     LOC.NAM       0.00      0.00      0.00         6
     LOC.NOM       0.00      0.00      0.00         6
     ORG.NAM       0.00      0.00      0.00        47
     ORG.NOM       0.00      0.00      0.00         5
     PER.NAM       0.00      0.00      0.00        89
     PER.NOM       0.00      0.00      0.00       208

   micro avg       0.00      0.00      0.00       388
   macro avg       0.00      0.00      0.00       388
weighted avg       0.00      0.00      0.00       388



验证中...: 100%|██████████| 9/9 [00:02<00:00,  3.70it/s]



[Epoch 5] F1:0.0025 P:0.0025 R:0.0026
              precision    recall  f1-score   support

     GPE.NAM       0.00      0.00      0.00        26
     GPE.NOM       0.00      0.00      0.00         1
     LOC.NAM       0.00      0.00      0.00         6
     LOC.NOM       0.00      0.00      0.00         6
     ORG.NAM       0.00      0.00      0.00        47
     ORG.NOM       0.00      0.00      0.00         5
     PER.NAM       0.00      0.00      0.00        89
     PER.NOM       0.00      0.00      0.00       208

   micro avg       0.00      0.00      0.00       388
   macro avg       0.00      0.00      0.00       388
weighted avg       0.00      0.00      0.00       388


训练完成！最佳验证集 F1：0.0039


In [9]:
model.load_state_dict(torch.load(conf.model_save_path, map_location=conf.device))
rep, f1, p, r = evaluate(model, test_loader)
print('=' * 50)
print(f'测试集 F1：{f1:.4f} | P：{p:.4f} | R：{r:.4f}')
print(rep)
print('达到 88% 目标！' if f1 >= 0.88 else f'当前 {f1:.4f}，可换 RoBERTa-wwm 模型提升')


验证中...: 100%|██████████| 9/9 [00:02<00:00,  4.09it/s]


测试集 F1：0.0000 | P：0.0000 | R：0.0000
              precision    recall  f1-score   support

     GPE.NAM       0.00      0.00      0.00        46
     GPE.NOM       0.00      0.00      0.00         2
     LOC.NAM       0.00      0.00      0.00        19
     LOC.NOM       0.00      0.00      0.00         9
     ORG.NAM       0.00      0.00      0.00        39
     ORG.NOM       0.00      0.00      0.00        16
     PER.NAM       0.00      0.00      0.00       112
     PER.NOM       0.00      0.00      0.00       169

   micro avg       0.00      0.00      0.00       412
   macro avg       0.00      0.00      0.00       412
weighted avg       0.00      0.00      0.00       412

当前 0.0000，可换 RoBERTa-wwm 模型提升


In [10]:
def predict(text):
    model.eval()
    chars = list(text)
    enc = conf.tokenizer(chars, is_split_into_words=True, truncation=True,
                         max_length=conf.max_len, return_tensors='pt')
    ids  = enc['input_ids'].to(conf.device)
    attn = enc['attention_mask'].to(conf.device)
    n    = min(len(chars), conf.max_len - 2)
    mask = torch.zeros(1, ids.shape[1], dtype=torch.bool, device=conf.device)
    mask[0, 1:n+1] = True
    with torch.no_grad():
        labs = [conf.id2label[i] for i in model.decode(ids, attn, mask)[0]]
    entities, buf, etype = [], [], None
    for c, l in zip(chars[:n], labs):
        if l.startswith('B-'):
            if buf: entities.append({'entity': ''.join(buf), 'type': etype})
            buf, etype = [c], l[2:]
        elif l.startswith('I-') and etype == l[2:]: buf.append(c)
        else:
            if buf: entities.append({'entity': ''.join(buf), 'type': etype})
            buf, etype = [], None
    if buf: entities.append({'entity': ''.join(buf), 'type': etype})
    return entities

for text in ['习近平在北京人民大会堂会见了特朗普', '阿里巴巴集团宣布在上海设立新总部', '人生如戏，导演是自己']:
    print(f'输入：{text}')
    for e in predict(text): print(f'  [{e["type"]}] {e["entity"]}')
    print()


输入：习近平在北京人民大会堂会见了特朗普
  [GPE.NAM] 京人
  [ORG.NOM] 民
  [GPE.NOM] 堂

输入：阿里巴巴集团宣布在上海设立新总部
  [PER.NOM] 团
  [PER.NOM] 宣

输入：人生如戏，导演是自己



In [11]:
from google.colab import files
files.download('saved/bert_crf_ner.pt')
print('下载完成！将 bert_crf_ner.pt 放到本地项目 saved/ 文件夹')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

下载完成！将 bert_crf_ner.pt 放到本地项目 saved/ 文件夹
